## Full historical extraction audit

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Locate and load the generated files
# ------------------------------------------------------------

repository_root = Path.cwd().parents[1]

trade_panel = pd.read_csv(
    repository_root
    / "data"
    / "processed"
    / "trade_total_country_year.csv"
)

mirror_values = pd.read_csv(
    repository_root
    / "data"
    / "processed"
    / "trade_total_mirror_values.csv"
)

coverage_report = pd.read_csv(
    repository_root
    / "outputs"
    / "tables"
    / "trade_coverage_by_year.csv"
)

validation_report = pd.read_csv(
    repository_root
    / "outputs"
    / "tables"
    / "trade_api_validation_report.csv"
)

source_report = pd.read_csv(
    repository_root
    / "outputs"
    / "tables"
    / "trade_value_source_by_year.csv"
)


# ------------------------------------------------------------
# 2. Define columns and remove the two anchor countries
# ------------------------------------------------------------

anchor_iso_codes = {"USA", "CHN"}

flow_columns = [
    "imports_from_china",
    "exports_to_china",
    "imports_from_us",
    "exports_to_us",
]

source_columns = [
    f"{relationship}_source"
    for relationship in flow_columns
]

scored_trade_panel = trade_panel.loc[
    ~trade_panel["reporter_iso"].isin(anchor_iso_codes)
].copy()


# ------------------------------------------------------------
# 3. Mirror recovery diagnostics
# ------------------------------------------------------------

mirror_values["mirror_found"] = (
    mirror_values["mirror_value_usd"].notna()
)

mirror_summary = pd.Series(
    {
        "mirror_checks": len(mirror_values),
        "mirror_values_found": int(
            mirror_values["mirror_found"].sum()
        ),
        "unresolved_mirror_checks": int(
            (~mirror_values["mirror_found"]).sum()
        ),
        "mirror_recovery_rate": (
            mirror_values["mirror_found"].mean()
        ),
    }
)

mirror_recovery_by_relationship = (
    mirror_values
    .groupby("missing_relationship")
    .agg(
        mirror_checks=("mirror_found", "size"),
        mirror_values_found=("mirror_found", "sum"),
    )
    .reset_index()
)

mirror_recovery_by_relationship["unresolved"] = (
    mirror_recovery_by_relationship["mirror_checks"]
    - mirror_recovery_by_relationship["mirror_values_found"]
)

mirror_recovery_by_relationship["recovery_rate"] = (
    mirror_recovery_by_relationship["mirror_values_found"]
    / mirror_recovery_by_relationship["mirror_checks"]
)


# ------------------------------------------------------------
# 4. Remaining missingness
# ------------------------------------------------------------

missing_by_flow = (
    scored_trade_panel[flow_columns]
    .isna()
    .sum()
    .rename("missing_observations")
    .reset_index()
    .rename(columns={"index": "relationship"})
)

unresolved_country_years = scored_trade_panel.loc[
    scored_trade_panel[flow_columns]
    .isna()
    .any(axis=1)
].copy()

unresolved_display_columns = [
    "year",
    "reporter_code",
    "reporter_iso",
    "reporter_name",
    *flow_columns,
    "available_flow_count",
]

unresolved_country_years = (
    unresolved_country_years[
        unresolved_display_columns
    ]
    .sort_values(
        ["year", "reporter_name"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Source provenance
# ------------------------------------------------------------

missing_source_columns = [
    column
    for column in source_columns
    if column not in scored_trade_panel.columns
]

if missing_source_columns:
    raise KeyError(
        "The following source columns are missing: "
        + ", ".join(missing_source_columns)
    )

source_summary = (
    scored_trade_panel[
        ["year", *source_columns]
    ]
    .melt(
        id_vars="year",
        value_vars=source_columns,
        var_name="relationship",
        value_name="value_source",
    )
    .groupby(
        ["relationship", "value_source"],
        dropna=False,
    )
    .size()
    .reset_index(name="observations")
)

allowed_sources = {
    "reporter",
    "mirror",
    "missing",
}

observed_sources = set(
    source_summary["value_source"].dropna()
)

unexpected_sources = (
    observed_sources - allowed_sources
)


# ------------------------------------------------------------
# 6. API extraction validation
# ------------------------------------------------------------

validation_summary = pd.Series(
    {
        "api_slices": len(validation_report),
        "duplicate_rows": int(
            validation_report["duplicate_rows"].sum()
        ),
        "missing_iso_rows": int(
            validation_report["missing_iso_rows"].sum()
        ),
        "minimum_slice_rows": int(
            validation_report["rows"].min()
        ),
        "maximum_slice_rows": int(
            validation_report["rows"].max()
        ),
    }
)


# ------------------------------------------------------------
# 7. Coverage diagnostics
# ------------------------------------------------------------

coverage_report["complete_share"] = (
    coverage_report["complete_four_flow_records"]
    / coverage_report["reporters_represented"]
)

coverage_summary = coverage_report[
    [
        "year",
        "reporters_represented",
        "complete_four_flow_records",
        "complete_share",
        "average_available_flows",
    ]
].copy()


# ------------------------------------------------------------
# 8. Score validation
# ------------------------------------------------------------

valid_scores = (
    scored_trade_panel["trade_orientation_score"]
    .dropna()
)

invalid_score_assignments = scored_trade_panel.loc[
    scored_trade_panel["trade_orientation_score"].notna()
    & (
        scored_trade_panel["available_flow_count"] < 4
    )
].copy()

score_summary = pd.Series(
    {
        "valid_trade_scores": len(valid_scores),
        "missing_trade_scores": int(
            scored_trade_panel[
                "trade_orientation_score"
            ].isna().sum()
        ),
        "minimum_trade_score": valid_scores.min(),
        "maximum_trade_score": valid_scores.max(),
        "scores_outside_range": int(
            (~valid_scores.between(-1, 1)).sum()
        ),
        "infinite_scores": int(
            np.isinf(valid_scores).sum()
        ),
        "scores_assigned_to_incomplete_records": (
            len(invalid_score_assignments)
        ),
    }
)


# ------------------------------------------------------------
# 9. Assertions for critical failures
# ------------------------------------------------------------

assert validation_summary["duplicate_rows"] == 0
assert validation_summary["missing_iso_rows"] == 0
assert score_summary["scores_outside_range"] == 0
assert score_summary["infinite_scores"] == 0
assert (
    score_summary[
        "scores_assigned_to_incomplete_records"
    ]
    == 0
)

if unexpected_sources:
    raise ValueError(
        "Unexpected source labels found: "
        f"{sorted(unexpected_sources)}"
    )


# ------------------------------------------------------------
# 10. Save detailed exception tables
# ------------------------------------------------------------

diagnostic_directory = (
    repository_root
    / "outputs"
    / "tables"
)

diagnostic_directory.mkdir(
    parents=True,
    exist_ok=True,
)

mirror_recovery_by_relationship.to_csv(
    diagnostic_directory
    / "trade_mirror_recovery_by_relationship.csv",
    index=False,
)

missing_by_flow.to_csv(
    diagnostic_directory
    / "trade_missing_by_flow.csv",
    index=False,
)

unresolved_country_years.to_csv(
    diagnostic_directory
    / "trade_unresolved_country_years.csv",
    index=False,
)

source_summary.to_csv(
    diagnostic_directory
    / "trade_source_provenance_summary.csv",
    index=False,
)


# ------------------------------------------------------------
# 11. Print one consolidated report
# ------------------------------------------------------------

print("=" * 70)
print("TRADE DATA DIAGNOSTIC REPORT")
print("=" * 70)

print("\nPANEL")
print(f"Rows: {len(trade_panel):,}")
print(
    "Years:",
    f"{trade_panel['year'].min()}-"
    f"{trade_panel['year'].max()}",
)
print(
    "Unique reporter codes:",
    trade_panel["reporter_code"].nunique(),
)

print("\nMIRROR RECOVERY")
print(
    mirror_summary.to_string(
        float_format=lambda value: f"{value:.3f}"
    )
)

print("\nMIRROR RECOVERY BY RELATIONSHIP")
print(
    mirror_recovery_by_relationship.to_string(
        index=False,
        formatters={
            "recovery_rate": lambda value: (
                f"{value:.1%}"
            )
        },
    )
)

print("\nREMAINING MISSING OBSERVATIONS")
print(
    missing_by_flow.to_string(index=False)
)

print("\nUNRESOLVED COUNTRY-YEARS")
print(
    f"Country-years: "
    f"{len(unresolved_country_years):,}"
)
print(
    "Affected reporters:",
    unresolved_country_years[
        "reporter_code"
    ].nunique(),
)
print(
    "Affected years:",
    unresolved_country_years[
        "year"
    ].nunique(),
)

print("\nAPI VALIDATION")
print(validation_summary.to_string())

print("\nSCORE VALIDATION")
print(
    score_summary.to_string(
        float_format=lambda value: f"{value:.6f}"
    )
)

print("\nCOVERAGE BY YEAR")
print(
    coverage_summary.to_string(
        index=False,
        formatters={
            "complete_share": lambda value: (
                f"{value:.1%}"
            ),
            "average_available_flows": (
                lambda value: f"{value:.2f}"
            ),
        },
    )
)

print("\nSOURCE PROVENANCE")
print(
    source_summary.to_string(index=False)
)

print("\nDetailed diagnostic tables saved to:")
print(diagnostic_directory)

print("\nAll critical assertions passed.")